In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Pretrained Swin Transformer Tiny Liver Cancer Multiclass MRI Classification Pipeline
========================================================
Google Colab compatible, PyTorch/Torchvision based.

Classes: Angiosarcoma / Cholangiocarcinoma / Healthy / Hemangioma / Hepatocellular Carcinoma

Includes:
- kaggle.json upload and automatic dataset download
- corrupted-image validation
- bilateral filtering + CLAHE preprocessing
- train-only augmentation
- ImageNet-1K pretrained Swin Transformer Tiny
- frozen-head training + final-stage fine-tuning
- Accuracy, Precision, Recall/Sensitivity, Specificity, F1, AUC
- train/validation/test metrics, confusion matrices, ROC curves
- bright accuracy/loss curves and box plots
- model complexity and training time
- XAI for first 20 test images: saliency, Swin feature attribution, LIME
- results ZIP + automatic Colab download
"""

import os, sys, json, time, shutil, random, zipfile, warnings, subprocess, importlib.util
from pathlib import Path
from typing import List, Optional
warnings.filterwarnings("ignore")

def install_if_missing(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        pkg = pip_name or import_name
        print(f"[INSTALL] {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for imp, pkg in [
    ("kaggle", "kaggle"), ("cv2", "opencv-python-headless"),
    ("torch", "torch"), ("torchvision", "torchvision"),
    ("sklearn", "scikit-learn"), ("lime", "lime"),
    ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("PIL", "Pillow")]:
    install_if_missing(imp, pkg)

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import swin_t, Swin_T_Weights

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, precision_recall_fscore_support,
    roc_auc_score, roc_curve, auc)
from sklearn.utils.class_weight import compute_class_weight
from lime import lime_image
from skimage.segmentation import mark_boundaries

SEED=42
IMG_SIZE=224
BATCH_SIZE=16
HEAD_EPOCHS=10
FINE_TUNE_EPOCHS=20
HEAD_LR=1e-3
FINE_TUNE_LR=2e-5
WEIGHT_DECAY=1e-4
TEST_SIZE=0.20
VAL_SIZE_FROM_REMAINING=0.20
N_XAI=20
LIME_NUM_SAMPLES=400

KAGGLE_KERNEL_REF=""
KAGGLE_DATASET_SLUG="ucimachinelearning/liver-cancer-multiclass-dataset"

WORK=Path("/content/liver_multiclass_swin_tiny_work") if Path("/content").exists() else Path.cwd()/"liver_multiclass_swin_tiny_work"
DATA=WORK/"dataset"
RESULTS=WORK/"results_swin_tiny"
MODEL_DIR=RESULTS/"model"
METRICS=RESULTS/"metrics"
PLOTS=RESULTS/"plots"
XAI=RESULTS/"xai"
SAMPLES=RESULTS/"preprocessing_samples"
for p in [WORK,DATA,RESULTS,MODEL_DIR,METRICS,PLOTS,XAI,SAMPLES]: p.mkdir(parents=True,exist_ok=True)

CLASS_ALIASES={
    "angiosarcoma":["angiosarcoma","angio_sarcoma"],
    "cholangiocarcinoma":["cholangiocarcinoma","cca"],
    "healthy":["healthy","normal","normal_liver"],
    "hemangioma":["hemangioma","haemangioma"],
    "hepatocellular_carcinoma":["hepatocellular_carcinoma","hepatocellular carcinoma","hcc","hepatoma"]
}
CLASS_ORDER=[
    "angiosarcoma",
    "cholangiocarcinoma",
    "healthy",
    "hemangioma",
    "hepatocellular_carcinoma"
]
EXTS={".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp"}
MEAN=[0.485,0.456,0.406]
STD=[0.229,0.224,0.225]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:",torch.__version__); print("Device:",DEVICE)

# ---------- Kaggle ----------
def in_colab():
    try:
        import google.colab
        return True
    except Exception: return False

def configure_kaggle():
    kd=Path.home()/".kaggle"; kd.mkdir(parents=True,exist_ok=True); target=kd/"kaggle.json"
    if target.exists(): os.chmod(target,0o600); return
    for c in [Path.cwd()/"kaggle.json",Path("/content/kaggle.json"),WORK/"kaggle.json"]:
        if c.exists(): shutil.copy2(c,target); os.chmod(target,0o600); return
    if in_colab():
        from google.colab import files
        print("Please upload kaggle.json...")
        up=files.upload(); names=[x for x in up if x.lower().endswith(".json")]
        if not names: raise FileNotFoundError("No kaggle.json uploaded")
        shutil.copy2(Path(names[0]),target); os.chmod(target,0o600)
    else: raise FileNotFoundError("kaggle.json not found")

def cmd(c):
    print("[CMD]"," ".join(c))
    return subprocess.run(c,check=True,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)

def parse_sources(meta):
    out=[]
    for f in list(meta.rglob("*metadata*.json"))+list(meta.rglob("kernel-metadata.json")):
        try: d=json.loads(f.read_text())
        except Exception: continue
        for k in ["dataset_sources","datasetSources"]:
            for x in d.get(k,[]) if isinstance(d.get(k,[]),list) else []:
                if isinstance(x,str) and "/" in x: out.append(x)
                elif isinstance(x,dict):
                    r=x.get("ref") or x.get("source") or x.get("dataset")
                    if isinstance(r,str) and "/" in r: out.append(r)
    return sorted(set(out))

def discover_sources():
    meta=WORK/"kaggle_kernel_metadata"
    if meta.exists(): shutil.rmtree(meta)
    meta.mkdir(parents=True,exist_ok=True)
    for c in [["kaggle","kernels","pull",KAGGLE_KERNEL_REF,"-p",str(meta),"-m"],
              ["kaggle","kernels","pull","-p",str(meta),"-m",KAGGLE_KERNEL_REF]]:
        try:
            r=cmd(c); print(r.stdout[-1000:]); s=parse_sources(meta); print("Sources:",s); return s
        except Exception: pass
    return []

def download_data():
    configure_kaggle()
    slugs=[KAGGLE_DATASET_SLUG] if KAGGLE_DATASET_SLUG.strip() else discover_sources()
    if not slugs: raise RuntimeError("Could not detect dataset; set KAGGLE_DATASET_SLUG")
    for slug in slugs:
        dest=DATA/slug.replace("/","__"); dest.mkdir(parents=True,exist_ok=True); mark=dest/".done"
        if mark.exists() and any(dest.rglob("*")): continue
        r=cmd(["kaggle","datasets","download","-d",slug,"-p",str(dest),"--unzip"])
        print(r.stdout[-1000:]); mark.touch()
    (RESULTS/"kaggle_sources.txt").write_text("\n".join(slugs))
    return slugs

# ---------- Data ----------
def nt(x): return x.lower().replace(" ","_").replace("-","_")
def infer_label(p):
    parts=[nt(x) for x in p.parts]
    for lab in CLASS_ORDER:
        aliases=[nt(a) for a in CLASS_ALIASES[lab]]
        for part in reversed(parts[:-1]):
            if part==lab or part in aliases: return lab
    joined="/".join(parts)
    for lab in ["malignant","benign","normal"]:
        for a in CLASS_ALIASES[lab]:
            if nt(a) in joined: return lab
    return None

def read_bgr(path):
    try:
        raw=np.fromfile(path,dtype=np.uint8)
        return cv2.imdecode(raw,cv2.IMREAD_COLOR) if raw.size else None
    except Exception: return None

def read_rgb(path):
    b=read_bgr(path)
    if b is None: raise ValueError(path)
    return cv2.cvtColor(b,cv2.COLOR_BGR2RGB)

def collect_images():
    rows=[]; bad=[]
    for p in DATA.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in EXTS: continue
        lab=infer_label(p)
        if lab is None: continue
        im=read_bgr(str(p))
        if im is None or im.ndim!=3 or min(im.shape[:2])<8: bad.append({"filepath":str(p),"label":lab})
        else: rows.append({"filepath":str(p),"label":lab})
    pd.DataFrame(bad).to_csv(METRICS/"skipped_corrupt_images.csv",index=False)
    df=pd.DataFrame(rows).drop_duplicates("filepath").reset_index(drop=True)
    if df.empty: raise RuntimeError("No valid images found")
    print(df.label.value_counts()); return df

def split_df(df):
    tv,te=train_test_split(df,test_size=TEST_SIZE,random_state=SEED,stratify=df.label)
    tr,va=train_test_split(tv,test_size=VAL_SIZE_FROM_REMAINING,random_state=SEED,stratify=tv.label)
    for n,d in [("training",tr),("validation",va),("test",te)]:
        d.to_csv(METRICS/f"{n}_split.csv",index=False); print(n,len(d)); print(d.label.value_counts())
    return tr.reset_index(drop=True),va.reset_index(drop=True),te.reset_index(drop=True)

def preprocess(rgb):
    g=cv2.cvtColor(np.asarray(rgb,np.uint8),cv2.COLOR_RGB2GRAY)
    g=cv2.bilateralFilter(g,9,75,75)
    g=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(g)
    return cv2.cvtColor(g,cv2.COLOR_GRAY2RGB)

def save_samples(df,n=9):
    s=df.sample(min(n,len(df)),random_state=SEED); fig,ax=plt.subplots(len(s),2,figsize=(8,max(6,3*len(s))))
    if len(s)==1: ax=np.array([ax])
    for i,(_,r) in enumerate(s.iterrows()):
        a=read_rgb(r.filepath); b=preprocess(a)
        ax[i,0].imshow(a); ax[i,0].set_title("Before: "+r.label); ax[i,0].axis("off")
        ax[i,1].imshow(b); ax[i,1].set_title("After: "+r.label); ax[i,1].axis("off")
    plt.tight_layout(); plt.savefig(SAMPLES/"before_after.png",dpi=300,bbox_inches="tight"); plt.close()

class LiverDataset(Dataset):
    def __init__(self,df,c2i,training=False):
        self.df=df.reset_index(drop=True); self.c2i=c2i
        aug=[transforms.Resize((IMG_SIZE,IMG_SIZE))]
        if training: aug += [transforms.RandomHorizontalFlip(),transforms.RandomVerticalFlip(0.2),transforms.RandomRotation(20),transforms.RandomAffine(0,translate=(0.06,0.06),scale=(0.92,1.08))]
        aug += [transforms.ToTensor(),transforms.Normalize(MEAN,STD)]
        self.tf=transforms.Compose(aug)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; im=Image.fromarray(preprocess(read_rgb(r.filepath)))
        return self.tf(im), self.c2i[r.label], r.filepath

# ---------- Model ----------
def build_model(nc):
    m=swin_t(weights=Swin_T_Weights.IMAGENET1K_V1)
    m.head=nn.Linear(m.head.in_features,nc)
    return m

def freeze_head(m):
    for p in m.parameters(): p.requires_grad=False
    for p in m.head.parameters(): p.requires_grad=True

def unfreeze_last(m,n=2):
    for p in m.parameters(): p.requires_grad=False
    feature_children=list(m.features.children())
    for module in feature_children[-max(1,n):]:
        for p in module.parameters(): p.requires_grad=True
    for p in m.norm.parameters(): p.requires_grad=True
    for p in m.head.parameters(): p.requires_grad=True

def specificity(cm):
    vals=[]; total=cm.sum()
    for i in range(cm.shape[0]):
        tp=cm[i,i]; fn=cm[i,:].sum()-tp; fp=cm[:,i].sum()-tp; tn=total-tp-fn-fp
        vals.append(tn/(tn+fp) if tn+fp else np.nan)
    return np.asarray(vals)

def metrics(y,p,classes):
    pred=p.argmax(1); cm=confusion_matrix(y,pred,labels=np.arange(len(classes)))
    try: au=roc_auc_score(label_binarize(y,classes=np.arange(len(classes))),p,average="macro",multi_class="ovr")
    except Exception: au=np.nan
    rec=recall_score(y,pred,average="macro",zero_division=0)
    return dict(accuracy=accuracy_score(y,pred),precision_macro=precision_score(y,pred,average="macro",zero_division=0),recall_macro=rec,sensitivity_macro=rec,specificity_macro=float(np.nanmean(specificity(cm))),f1_macro=f1_score(y,pred,average="macro",zero_division=0),auc_macro_ovr=au)

def run_epoch(m,loader,crit,opt=None):
    train=opt is not None; m.train(train); total=0; ys=[]; ps=[]
    for x,y,_ in loader:
        x=x.to(DEVICE); y=y.to(DEVICE)
        if train: opt.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train):
            logits=m(x); loss=crit(logits,y)
            if train: loss.backward(); opt.step()
        total += loss.item()*len(y); ys.append(y.detach().cpu().numpy()); ps.append(torch.softmax(logits.detach(),1).cpu().numpy())
    return total/len(loader.dataset),np.concatenate(ys),np.concatenate(ps)

def train_model(m,tr,va,te,classes,cw):
    crit=nn.CrossEntropyLoss(weight=torch.tensor(cw,dtype=torch.float32,device=DEVICE))
    best=1e9; bestp=MODEL_DIR/"best_swin_tiny.pth"; hist=[]; epoch=0
    stages=[("head",HEAD_EPOCHS,HEAD_LR,freeze_head),("finetune",FINE_TUNE_EPOCHS,FINE_TUNE_LR,lambda z:unfreeze_last(z,2))]
    for stage,epochs,lr,setup in stages:
        setup(m); opt=torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],lr=lr,weight_decay=WEIGHT_DECAY)
        sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="min",factor=0.2,patience=3); patience=0
        for _ in range(epochs):
            epoch+=1; tl,yt,pt=run_epoch(m,tr,crit,opt); vl,yv,pv=run_epoch(m,va,crit); tsl,ys,ps=run_epoch(m,te,crit)
            mt,mv,ms=metrics(yt,pt,classes),metrics(yv,pv,classes),metrics(ys,ps,classes); sch.step(vl)
            hist.append({"epoch":epoch,"stage":stage,"train_loss":tl,"val_loss":vl,"test_loss":tsl,**{f"train_{k}":v for k,v in mt.items()},**{f"val_{k}":v for k,v in mv.items()},**{f"test_{k}":v for k,v in ms.items()}})
            print(f"Epoch {epoch}: train={mt['accuracy']:.4f} val={mv['accuracy']:.4f} test={ms['accuracy']:.4f} val_loss={vl:.4f}")
            if vl<best: best=vl; patience=0; torch.save(m.state_dict(),bestp)
            else: patience+=1
            if patience>=8: break
    pd.DataFrame(hist).to_csv(METRICS/"training_history.csv",index=False)
    m.load_state_dict(torch.load(bestp,map_location=DEVICE,weights_only=True)); return pd.DataFrame(hist)

def evaluate(m,loader,name,classes):
    loss,y,p=run_epoch(m,loader,nn.CrossEntropyLoss()); pred=p.argmax(1); cm=confusion_matrix(y,pred,labels=np.arange(len(classes)))
    ov=metrics(y,p,classes); ov.update(split=name,loss=loss,n_samples=len(y)); pd.DataFrame([ov]).to_csv(METRICS/f"{name}_overall_metrics.csv",index=False)
    pr,re,f1,sup=precision_recall_fscore_support(y,pred,labels=np.arange(len(classes)),zero_division=0); sp=specificity(cm); yb=label_binarize(y,classes=np.arange(len(classes))); rows=[]
    for i,c in enumerate(classes):
        tp=cm[i,i]; fn=cm[i,:].sum()-tp; fp=cm[:,i].sum()-tp; tn=cm.sum()-tp-fn-fp
        try: au=roc_auc_score(yb[:,i],p[:,i])
        except Exception: au=np.nan
        rows.append(dict(split=name,class_name=c,accuracy_ovr=(tp+tn)/cm.sum(),precision=pr[i],recall_sensitivity=re[i],specificity=sp[i],f1_score=f1[i],auc_ovr=au,support=sup[i]))
    pc=pd.DataFrame(rows); pc.to_csv(METRICS/f"{name}_per_class_metrics.csv",index=False)
    pd.DataFrame(classification_report(y,pred,labels=np.arange(len(classes)),target_names=classes,output_dict=True,zero_division=0)).T.to_csv(METRICS/f"{name}_classification_report.csv")
    return ov,y,pred,p,pc

# ---------- Plots ----------
def plot_history(h):
    plt.figure(figsize=(9,6)); plt.plot(h.epoch,h.train_accuracy,color="#00BFFF",label="Training",linewidth=2.5); plt.plot(h.epoch,h.val_accuracy,color="#FF1493",label="Validation",linewidth=2.5); plt.plot(h.epoch,h.test_accuracy,color="#32CD32",label="Test",linewidth=2.5); plt.ylim(0,1.05); plt.legend(); plt.grid(alpha=.25); plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Pretrained Swin Transformer Tiny Accuracy Curves"); plt.tight_layout(); plt.savefig(PLOTS/"accuracy_curves.png",dpi=300); plt.close()
    plt.figure(figsize=(9,6)); plt.plot(h.epoch,h.train_loss,color="#FF8C00",label="Training",linewidth=2.5); plt.plot(h.epoch,h.val_loss,color="#9400D3",label="Validation",linewidth=2.5); plt.plot(h.epoch,h.test_loss,color="#00CED1",label="Test",linewidth=2.5); plt.legend(); plt.grid(alpha=.25); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Pretrained Swin Transformer Tiny Loss Curves"); plt.tight_layout(); plt.savefig(PLOTS/"loss_curves.png",dpi=300); plt.close()

def plot_eval(y,pred,prob,pc,classes,name):
    cm=confusion_matrix(y,pred,labels=np.arange(len(classes))); fig,ax=plt.subplots(figsize=(7,6)); im=ax.imshow(cm,cmap="turbo"); plt.colorbar(im,ax=ax); ax.set_xticks(range(len(classes)),classes,rotation=35); ax.set_yticks(range(len(classes)),classes); ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(name.title()+" Confusion Matrix")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]): ax.text(j,i,str(cm[i,j]),ha="center",va="center")
    plt.tight_layout(); plt.savefig(PLOTS/f"{name}_confusion_matrix.png",dpi=300); plt.close()
    yb=label_binarize(y,classes=np.arange(len(classes))); plt.figure(figsize=(8,7))
    for i,c in enumerate(classes):
        try: fpr,tpr,_=roc_curve(yb[:,i],prob[:,i]); plt.plot(fpr,tpr,label=f"{c} AUC={auc(fpr,tpr):.3f}",linewidth=2.2)
        except Exception: pass
    plt.plot([0,1],[0,1],"k--"); plt.legend(); plt.grid(alpha=.25); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(name.title()+" ROC-AUC"); plt.tight_layout(); plt.savefig(PLOTS/f"{name}_roc_auc.png",dpi=300); plt.close()
    cols=["accuracy_ovr","precision","recall_sensitivity","specificity","f1_score","auc_ovr"]; data=[pc[c].dropna().values for c in cols]; plt.figure(figsize=(10,6)); plt.boxplot(data,labels=["Accuracy","Precision","Recall","Specificity","F1","AUC"],patch_artist=True,showmeans=True); plt.ylim(0,1.05); plt.grid(axis="y",alpha=.25); plt.title(name.title()+" Metric Box Plot"); plt.tight_layout(); plt.savefig(PLOTS/f"{name}_metric_boxplot.png",dpi=300); plt.close()

# ---------- XAI ----------
def xai_tensor(rgb):
    tfm=transforms.Compose([
        transforms.Resize((IMG_SIZE,IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN,STD)
    ])
    return tfm(Image.fromarray(preprocess(rgb)))

def normmap(x):
    x=np.nan_to_num(np.asarray(x,np.float32)); x-=x.min(); return x/(x.max()+1e-8)

def overlay(rgb,h):
    h=cv2.resize(normmap(h),(rgb.shape[1],rgb.shape[0]))
    c=cv2.cvtColor(cv2.applyColorMap(np.uint8(h*255),cv2.COLORMAP_JET),cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(rgb.astype(np.uint8),.55,c,.45,0)

def input_saliency(m,t,pred):
    x=t.unsqueeze(0).to(DEVICE); x.requires_grad_(True); m.eval(); m.zero_grad(set_to_none=True)
    m(x)[0,pred].backward()
    if x.grad is None: raise RuntimeError('Input gradient unavailable')
    return normmap(x.grad.detach()[0].abs().max(0).values.cpu().numpy())

def swin_feature_attribution(m,t,pred):
    captured={}
    target=list(m.features.children())[-1]
    def hook(module,inputs,output):
        if torch.is_tensor(output): output.retain_grad()
        captured['out']=output
    h=target.register_forward_hook(hook)
    try:
        x=t.unsqueeze(0).to(DEVICE); m.eval(); m.zero_grad(set_to_none=True)
        m(x)[0,pred].backward(); out=captured.get('out')
        if out is None or out.grad is None: raise RuntimeError('Swin feature gradient unavailable')
        a=out.detach()[0]; g=out.grad.detach()[0]
        if a.ndim==3:
            if a.shape[-1]>=max(a.shape[0],a.shape[1]): cam=(a*g).sum(-1)
            else: cam=(a*g).sum(0)
        elif a.ndim==2:
            s=(a*g).sum(-1); side=int(np.sqrt(s.numel()))
            if side*side!=s.numel(): raise RuntimeError('Cannot reshape Swin attribution')
            cam=s.reshape(side,side)
        else: raise RuntimeError(f'Unsupported Swin feature shape {tuple(a.shape)}')
        return normmap(torch.relu(cam).cpu().numpy())
    finally:
        h.remove()

def generate_xai(m,test_df,classes,c2i):
    for d in ['saliency','swin_feature_attribution','lime','combined']:
        (XAI/d).mkdir(parents=True,exist_ok=True)
    def predict(images):
        batch=torch.stack([xai_tensor(np.asarray(im,np.uint8)) for im in images]).to(DEVICE)
        with torch.no_grad(): return torch.softmax(m(batch),1).cpu().numpy()
    expl=lime_image.LimeImageExplainer(random_state=SEED); rows=[]
    for k,(_,r) in enumerate(test_df.head(N_XAI).iterrows(),1):
        print(f'Swin XAI {k}/{min(N_XAI,len(test_df))}: {r.filepath}')
        rgb=cv2.resize(read_rgb(r.filepath),(IMG_SIZE,IMG_SIZE)); t=xai_tensor(rgb)
        with torch.no_grad(): prob=torch.softmax(m(t.unsqueeze(0).to(DEVICE)),1)[0].cpu().numpy()
        pred=int(prob.argmax()); true=c2i[r.label]; stem=f'Patient_{k:02d}_true_{classes[true]}_pred_{classes[pred]}'
        salim=rgb.copy(); featim=rgb.copy(); limeim=rgb.copy()
        try:
            salim=overlay(rgb,input_saliency(m,t,pred)); cv2.imwrite(str(XAI/'saliency'/f'{stem}.png'),cv2.cvtColor(salim,cv2.COLOR_RGB2BGR))
        except Exception as ex: print('Saliency failed:',ex)
        try:
            featim=overlay(rgb,swin_feature_attribution(m,t,pred)); cv2.imwrite(str(XAI/'swin_feature_attribution'/f'{stem}.png'),cv2.cvtColor(featim,cv2.COLOR_RGB2BGR))
        except Exception as ex: print('Swin attribution failed:',ex)
        try:
            e=expl.explain_instance(rgb.astype(float),predict,top_labels=len(classes),hide_color=0,num_samples=min(LIME_NUM_SAMPLES,120),random_seed=SEED)
            temp,mask=e.get_image_and_mask(pred,positive_only=True,num_features=10,hide_rest=False)
            limeim=np.uint8(np.clip(mark_boundaries(np.clip(temp,0,255)/255.,mask),0,1)*255)
            cv2.imwrite(str(XAI/'lime'/f'{stem}.png'),cv2.cvtColor(limeim,cv2.COLOR_RGB2BGR))
        except Exception as ex: print('LIME failed:',ex)
        fig,ax=plt.subplots(1,4,figsize=(18,5))
        for a,im,title in zip(ax,[rgb,salim,featim,limeim],['Original','Saliency','Swin Feature Attribution','LIME']): a.imshow(im); a.set_title(title); a.axis('off')
        fig.suptitle(f'True: {classes[true]} | Pred: {classes[pred]} ({prob[pred]:.4f})'); plt.tight_layout(); plt.savefig(XAI/'combined'/f'{stem}.png',dpi=250); plt.close()
        rec={'patient_number':k,'filepath':r.filepath,'true_class':classes[true],'predicted_class':classes[pred],'confidence':float(prob[pred]),'correct':int(true==pred)}
        rec.update({f'prob_{c}':float(prob[i]) for i,c in enumerate(classes)}); rows.append(rec)
    pd.DataFrame(rows).to_csv(XAI/'first_20_test_xai_predictions.csv',index=False)

# ---------- Main ----------
def main():
    start=time.perf_counter(); slugs=download_data(); df=collect_images(); classes=[c for c in CLASS_ORDER if c in df.label.unique()]; c2i={c:i for i,c in enumerate(classes)}; (RESULTS/"class_mapping.json").write_text(json.dumps(c2i,indent=2)); tr,va,te=split_df(df); save_samples(tr)
    dtr=LiverDataset(tr,c2i,True); dtre=LiverDataset(tr,c2i,False); dva=LiverDataset(va,c2i,False); dte=LiverDataset(te,c2i,False)
    kwargs=dict(batch_size=BATCH_SIZE,num_workers=2,pin_memory=DEVICE.type=="cuda"); ltr=DataLoader(dtr,shuffle=True,**kwargs); ltre=DataLoader(dtre,shuffle=False,**kwargs); lva=DataLoader(dva,shuffle=False,**kwargs); lte=DataLoader(dte,shuffle=False,**kwargs)
    y0=np.array([c2i[x] for x in tr.label]); cw=compute_class_weight(class_weight="balanced",classes=np.arange(len(classes)),y=y0); print("Class weights:",cw)
    print("Loading torchvision ImageNet-1K pretrained Swin Transformer Tiny..."); m=build_model(len(classes)).to(DEVICE); print("Parameters:",sum(p.numel() for p in m.parameters()))
    ts=time.perf_counter(); h=train_model(m,ltr,lva,lte,classes,cw); training_time=time.perf_counter()-ts; plot_history(h); allm=[]
    for name,loader in [("training",ltre),("validation",lva),("test",lte)]:
        ov,y,pred,prob,pc=evaluate(m,loader,name,classes); allm.append(ov); plot_eval(y,pred,prob,pc,classes,name)
    pd.DataFrame(allm).to_csv(METRICS/"all_split_overall_metrics.csv",index=False); print(pd.DataFrame(allm).to_string(index=False))
    total=sum(p.numel() for p in m.parameters()); trainable=sum(p.numel() for p in m.parameters() if p.requires_grad); comp={"model":"Pretrained Swin Transformer Tiny","weights":"Swin_T_Weights.IMAGENET1K_V1","input_size":224,"architecture_family":"hierarchical shifted-window transformer","total_parameters":total,"trainable_parameters_after_finetuning":trainable,"non_trainable_parameters_after_finetuning":total-trainable,"reference_gflops":4.5,"training_time_seconds":training_time,"training_time_minutes":training_time/60}; pd.DataFrame([comp]).to_csv(METRICS/"model_complexity_and_time.csv",index=False); (METRICS/"model_complexity_and_time.json").write_text(json.dumps(comp,indent=2)); torch.save({"model_state_dict":m.state_dict(),"classes":classes,"class_to_idx":c2i,"architecture":"swin_t","weights":"IMAGENET1K_V1"},MODEL_DIR/"final_pretrained_swin_tiny.pth")
    safety=WORK/"Swin_Tiny_RESULTS_BEFORE_XAI.zip"; safety.unlink(missing_ok=True)
    with zipfile.ZipFile(safety,"w",zipfile.ZIP_DEFLATED) as zf:
        for p in RESULTS.rglob("*"):
            if p.is_file(): zf.write(p,arcname=p.relative_to(RESULTS.parent))
    print("Safety ZIP:",safety)
    try:
        generate_xai(m,te,classes,c2i)
    except Exception as ex:
        print("XAI stopped safely:",type(ex).__name__,ex)
        (XAI/"XAI_ERROR.txt").write_text(f"{type(ex).__name__}: {ex}")
    meta={"model":"Pretrained Swin Transformer Tiny","framework":"PyTorch/Torchvision","weights":"Swin_T_Weights.IMAGENET1K_V1","classes":classes,"kaggle_sources":slugs,"total_pipeline_seconds":time.perf_counter()-start}; (RESULTS/"run_metadata.json").write_text(json.dumps(meta,indent=2))
    z=WORK/"Pretrained_Swin_Tiny_Liver_Cancer_Multiclass_Results.zip"; z.unlink(missing_ok=True)
    with zipfile.ZipFile(z,"w",zipfile.ZIP_DEFLATED) as f:
        for p in RESULTS.rglob("*"):
            if p.is_file(): f.write(p,arcname=p.relative_to(RESULTS.parent))
    print("ZIP:",z)
    if in_colab():
        from google.colab import files
        files.download(str(z))

if __name__=="__main__": main()


[INSTALL] lime
PyTorch: 2.11.0+cu128
Device: cuda
Please upload kaggle.json...


Saving kaggle.json to kaggle.json
[CMD] kaggle datasets download -d ucimachinelearning/liver-cancer-multiclass-dataset -p /content/liver_multiclass_swin_tiny_work/dataset/ucimachinelearning__liver-cancer-multiclass-dataset --unzip
/602M [00:26<00:02, 23.6MB/s]
 90%|████████▉ | 541M/602M [00:26<00:02, 22.8MB/s]
 91%|█████████ | 545M/602M [00:27<00:02, 23.2MB/s]
 91%|█████████ | 549M/602M [00:27<00:02, 21.9MB/s]
 92%|█████████▏| 552M/602M [00:27<00:02, 20.3MB/s]
 92%|█████████▏| 554M/602M [00:27<00:02, 19.4MB/s]
 93%|█████████▎| 558M/602M [00:27<00:02, 21.0MB/s]
 93%|█████████▎| 562M/602M [00:27<00:01, 22.2MB/s]
 94%|█████████▍| 565M/602M [00:28<00:01, 21.7MB/s]
 95%|█████████▍| 569M/602M [00:28<00:01, 22.6MB/s]
 95%|█████████▌| 572M/602M [00:28<00:01, 20.7MB/s]
 96%|█████████▌| 576M/602M [00:28<00:01, 21.6MB/s]
 96%|█████████▌| 579M/602M [00:28<00:01, 21.3MB/s]
 97%|█████████▋| 583M/602M [00:28<00:00, 22.3MB/s]
 97%|█████████▋| 587M/602M [00:29<00:00, 23.1MB/s]
 98%|█████████▊| 590M/602

100%|██████████| 108M/108M [00:00<00:00, 137MB/s]


Parameters: 27523199
Epoch 1: train=0.6016 val=0.6873 test=0.6836 val_loss=0.6964
Epoch 2: train=0.7219 val=0.8431 test=0.8533 val_loss=0.4421
Epoch 3: train=0.7777 val=0.8489 test=0.8408 val_loss=0.4238
Epoch 4: train=0.7949 val=0.8286 test=0.8299 val_loss=0.4083
Epoch 5: train=0.8092 val=0.8791 test=0.8766 val_loss=0.3287
Epoch 6: train=0.8137 val=0.8895 test=0.8862 val_loss=0.3059
Epoch 7: train=0.8238 val=0.8989 test=0.8950 val_loss=0.2899
Epoch 8: train=0.8311 val=0.9130 test=0.9158 val_loss=0.2546
Epoch 9: train=0.8421 val=0.9208 test=0.9204 val_loss=0.2250
Epoch 10: train=0.8495 val=0.9140 test=0.9254 val_loss=0.2354
Epoch 11: train=0.9034 val=0.9891 test=0.9858 val_loss=0.0400
Epoch 12: train=0.9361 val=0.9885 test=0.9829 val_loss=0.0343
Epoch 13: train=0.9510 val=0.9953 test=0.9958 val_loss=0.0137
Epoch 14: train=0.9613 val=0.9969 test=0.9975 val_loss=0.0072
Epoch 15: train=0.9662 val=0.9990 test=0.9971 val_loss=0.0069
Epoch 16: train=0.9712 val=0.9990 test=0.9996 val_loss=0.0

  0%|          | 0/120 [00:00<?, ?it/s]

...

Swin XAI 20/20: /content/liver_multiclass_swin_tiny_work/dataset/ucimachinelearning__liver-cancer-multiclass-dataset/Liver_Dataset/Hemangioma/Hemangioma_0732.jpg


  0%|          | 0/120 [00:00<?, ?it/s]

ZIP: /content/liver_multiclass_swin_tiny_work/Pretrained_Swin_Tiny_Liver_Cancer_Multiclass_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>